## loading excel file

In [8]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_excel("daily_activity_log.xlsx")

## sanity check

In [10]:
# loaded correctly?
df.head()

,record_id,user_id,date,day_of_week,activity_type,planned_time_hrs,actual_time_hrs,completion_status,interruption_count,energy_level,mood,remarks
0,1,101,2025-11-01,Mon,study,3.0,2.5,yes,2,4.0,High,good focus
1,2,101,2025-11-02,Tue,Study,2.0,3.0,yes,5,3.0,neutral,overstudied
2,3,101,2025-11-03,Wed,studdy,3.0,1.5,no,7,NaN,low,lack of focus
3,4,101,2025-11-04,Thu,work,4.0,4.5,yes,1,5.0,HIGH,productive day
4,5,101,2025-11-05,Fri,rest,1.0,0.5,no,3,2.0,Low,tired


In [11]:
# structure and null profile

df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   record_id           30 non-null     int64         
 1   user_id             30 non-null     int64         
 2   date                30 non-null     datetime64[ns]
 3   day_of_week         30 non-null     object        
 4   activity_type       30 non-null     object        
 5   planned_time_hrs    30 non-null     float64       
 6   actual_time_hrs     30 non-null     float64       
 7   completion_status   30 non-null     object        
 8   interruption_count  30 non-null     int64         
 9   energy_level        26 non-null     float64       
 10  mood                30 non-null     object        
 11  remarks             30 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(3), object(5)
memory usage: 2.9+ KB


record_id             0
user_id               0
date                  0
day_of_week           0
activity_type         0
planned_time_hrs      0
actual_time_hrs       0
completion_status     0
interruption_count    0
energy_level          4
mood                  0
remarks               0
dtype: int64

#  DATA EXPANSION

In [12]:
# ---------- DATA EXPANSION ----------

np.random.seed(42)

expanded_dfs = [df]

for i in range(1, 4):  # 3 extra months approx
    temp = df.copy()

    # shift dates forward
    temp['date'] = temp['date'] + pd.to_timedelta(7 * i, unit='D')

    # add slight randomness
    temp['planned_time_hrs'] += np.random.choice([0, 0.5, -0.5], size=len(temp))
    temp['actual_time_hrs'] += np.random.choice([0, 0.5, -0.5], size=len(temp))

    # randomly introduce missing energy
    mask = np.random.rand(len(temp)) < 0.1
    temp.loc[mask, 'energy_level'] = np.nan

    expanded_dfs.append(temp)

df = pd.concat(expanded_dfs, ignore_index=True)


In [13]:
# checkpoint

df.shape

(120, 12)

# categorical cleaning

In [15]:
df['activity_type'] = (
    df['activity_type']
    .str.strip()
    .str.lower()
)

df['activity_type'] = df['activity_type'].replace({
    'studdy': 'study'
})

# quick check
df['activity_type'].value_counts()

activity_type
study    72
work     28
rest     20
Name: count, dtype: int64

# normalize other text field and invalid time handling

In [19]:
# normalize other text field 

text_cols = ['day_of_week', 'completion_status', 'mood']

for col in text_cols:
    df[col] = df[col].str.strip().str.lower()

In [20]:
# invalid time handling (auditable)

df['invalid_time_flag'] = df['actual_time_hrs'] < 0

In [21]:
# check
df[df['invalid_time_flag']]

,record_id,user_id,date,day_of_week,activity_type,planned_time_hrs,actual_time_hrs,completion_status,interruption_count,energy_level,mood,remarks,invalid_time_flag
25,26,101,2025-11-13,sat,study,3.0,-1.0,no,9,2.0,low,data error,True
55,26,101,2025-11-20,sat,study,3.0,-1.5,no,9,NaN,low,data error,True
82,23,101,2025-11-25,thu,rest,0.5,-0.5,no,5,1.0,low,exhausted,True
85,26,101,2025-11-27,sat,study,2.5,-1.5,no,9,2.0,low,data error,True
115,26,101,2025-12-04,sat,study,2.5,-1.0,no,9,NaN,low,data error,True


In [22]:
# fix
df.loc[df['actual_time_hrs'] < 0, 'actual_time_hrs'] = np.nan

# energy imputation & duplicate flagging

In [31]:
df['energy_missing_flag'] = df['energy_level'].isnull()

df['energy_level'] = (
    df.groupby('user_id')['energy_level']
      .transform(lambda x: x.fillna(x.median()))
)

In [25]:
# check 
df['energy_level'].isnull().sum()

np.int64(0)

In [27]:
# duplicate flagging

df['duplicate_flag'] = df.duplicated(
    subset=['user_id', 'date', 'activity_type'],
    keep=False
)

# FEATURE ENGINEERING

In [28]:
df['efficiency'] = df['actual_time_hrs'] / df['planned_time_hrs']

df['burnout_risk'] = np.where(
    (df['energy_level'] <= 2) &
    (df['actual_time_hrs'] > df['planned_time_hrs']),
    1, 0
)

df['high_interruption_flag'] = df['interruption_count'] >= 5

# final export

In [29]:
df.to_csv("cleaned_activity_data.csv", index=False)